# 🧮 Math Escalation — IMPROVED Training (Target: 5/5 Eval)

**Previous score: 2/5. This notebook targets 5/5 by:**
- More training steps (200 steps vs 20)
- Cleaner answer extraction with `<answer>` tags
- Stronger format reward signal
- Smarter prompt that explicitly shows expected format
- Better curriculum (start easy, build up)

In [ ]:
# Cell 1 — Install
!pip install "unsloth[colab-new]" trl>=0.12.0 datasets -q
print('✅ Done!')

In [ ]:
# Cell 2 — Embedded Math Environment
import random, re, math

def generate_problem(difficulty: int):
    d = min(difficulty, 10)
    if d == 1:
        a, b = random.randint(1, 9), random.randint(1, 9)
        return f"{a} + {b}", float(a + b)
    elif d == 2:
        a, b = random.randint(10, 50), random.randint(10, 50)
        return f"{a} + {b}", float(a + b)
    elif d == 3:
        a, b = sorted([random.randint(10, 99), random.randint(1, 50)], reverse=True)
        return f"{a} - {b}", float(a - b)
    elif d == 4:
        a, b = random.randint(2, 9), random.randint(2, 9)
        return f"{a} * {b}", float(a * b)
    elif d == 5:
        a, b, c = random.randint(2, 8), random.randint(2, 8), random.randint(1, 15)
        return f"{a} * {b} + {c}", float(a * b + c)
    elif d == 6:
        a, b, c = random.randint(2, 8), random.randint(2, 8), random.randint(1, 10)
        return f"{a} * ({b} + {c})", float(a * (b + c))
    elif d == 7:
        b = random.randint(2, 9)
        a = random.randint(2, 12) * b
        return f"{a} / {b}", float(a // b)
    elif d == 8:
        a, x = random.randint(2, 9), random.randint(1, 10)
        b = random.randint(1, 20)
        c = a * x + b
        return f"Solve for x: {a}x + {b} = {c}", float(x)
    elif d == 9:
        x = random.randint(2, 12)
        return f"sqrt({x * x})", float(x)
    else:
        a = random.randint(2, 4)
        x = random.randint(1, 8)
        div = random.randint(2, 4)
        b = div * random.randint(2, 6) - a * x
        e = (a * x + b) // div
        return f"Solve for x: ({a}x + {b}) / {div} = {e}", float(x)

print('✅ Math environment ready (Tiers 1-10)')

In [ ]:
# Cell 3 — Load Model with Unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen2.5-1.5B-Instruct',
    max_seq_length = 512,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 42,
)
print(f'✅ Model ready: {model.num_parameters():,} params')

In [ ]:
# Cell 4 — IMPROVED Reward Function
# Key insight: reward BOTH <thought> AND <answer>N</answer> tags
# This makes answer extraction deterministic and clean

def extract_answer(text):
    """Try to extract number from <answer>N</answer> tag, then fallback to last number."""
    # Primary: structured tag
    m = re.search(r'<answer>\s*(-?\d+\.?\d*)\s*</answer>', text)
    if m:
        return float(m.group(1)), True  # (value, was_structured)
    # Fallback: last number in text (after removing thought tags)
    clean = re.sub(r'<thought>.*?</thought>', '', text, flags=re.DOTALL)
    nums = re.findall(r'-?\d+\.?\d*', clean)
    if nums:
        return float(nums[-1]), False
    return None, False

def compute_reward(completions, prompts, answers, **kwargs):
    """
    Multi-component reward. Signals:
      +0.6  using <thought> tags with substance  (format reward)
      +0.4  using <answer>N</answer> tag         (structured output)
      +1.0  correct answer                       (correctness)
      -0.3  wrong answer                         (no format penalty on wrong)
    """
    rewards = []
    for completion, expected_str in zip(completions, answers):
        text = completion[0]['content'] if isinstance(completion, list) else str(completion)
        expected = float(expected_str)
        
        total = 0.0
        
        # Reward 1: Reasoning tags
        if '<thought>' in text and '</thought>' in text:
            m = re.search(r'<thought>(.*?)</thought>', text, re.DOTALL)
            if m and len(m.group(1).strip()) > 5:
                total += 0.6
        
        # Reward 2: Structured answer tag
        if '<answer>' in text and '</answer>' in text:
            total += 0.4
        
        # Reward 3: Correctness
        predicted, is_structured = extract_answer(text)
        if predicted is not None:
            if abs(predicted - expected) < 1e-4:
                total += 1.0  # Correct!
            else:
                total -= 0.3  # Wrong answer penalty
        else:
            total -= 0.5  # No number at all
        
        rewards.append(total)
    
    return rewards

print('✅ Reward function ready (max possible = 2.0)')

In [ ]:
# Cell 5 — Build Dataset with EXPLICIT Format Example in Prompt
import datasets

# System message teaches the model EXACTLY what format to use
SYSTEM = (
    'You are a math solver. ALWAYS respond in this EXACT format:\n'
    '<thought>\n[your step-by-step reasoning here]\n</thought>\n'
    '<answer>[ONLY the numeric answer, nothing else]</answer>\n\n'
    'Example:\nProblem: 3 + 5\n'
    '<thought>\n3 + 5 = 8\n</thought>\n'
    '<answer>8</answer>'
)

def make_prompt(problem_str, tier):
    # Use chat template for best results with instruction-tuned model
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': f'Problem: {problem_str}'}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def build_dataset(n=800):
    print(f'Building dataset with {n} problems...')
    # Heavily weight easy tiers to ensure model learns the format first
    tier_weights = {1:200, 2:150, 3:100, 4:100, 5:80, 6:60, 7:50, 8:30, 9:20, 10:10}
    data = []
    for tier, count in tier_weights.items():
        for _ in range(count):
            prob, ans = generate_problem(tier)
            data.append({'prompt': make_prompt(prob, tier), 'answer': str(ans)})
    
    random.shuffle(data)
    ds = datasets.Dataset.from_list(data)
    print(f'✅ Dataset: {len(ds)} problems')
    print(f"\nSample prompt:\n{ds[0]['prompt'][-300:]}")
    return ds

train_dataset = build_dataset(800)

In [ ]:
# Cell 6 — GRPO Training (200 steps for real learning)
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir = './math_grpo_v2',
    num_train_epochs = 3,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 8,
    learning_rate = 8e-6,
    max_steps = 200,
    logging_steps = 10,
    save_steps = 200,
    warmup_ratio = 0.05,
    lr_scheduler_type = 'cosine',
    report_to = 'none',
    # GRPO
    num_generations = 8,
    max_completion_length = 200,
    temperature = 1.0,
    # Memory
    gradient_checkpointing = True,
)

trainer = GRPOTrainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    reward_funcs = compute_reward,
    processing_class = tokenizer,
)

print('🔥 Starting training (200 steps, ~45-60 min on T4)...')
trainer.train()
print('✅ Training done!')

In [ ]:
# Cell 7 — Save Merged Model
model.save_pretrained_merged('./math_grpo_v2_merged', tokenizer, save_method='merged_16bit')
print('💾 Saved to ./math_grpo_v2_merged')

In [ ]:
# Cell 8 — EVALUATION (Same 5 test cases as before)
FastLanguageModel.for_inference(model)

def evaluate(problem_str, expected):
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': f'Problem: {problem_str}'}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=True,
            repetition_penalty=1.1
        )
    response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    predicted, is_structured = extract_answer(response)
    correct = predicted is not None and abs(predicted - expected) < 1e-4
    return response, predicted, correct, is_structured

# The 5 test cases
test_cases = [
    ('5 + 7',       12.0),
    ('45 + 38',     83.0),
    ('6 * 8',       48.0),
    ('3 * (4 + 5)', 27.0),
    ('sqrt(49)',     7.0),
]

print('='*55)
print('📊 EVALUATION RESULTS')
print('='*55)
score = 0
for prob, expected in test_cases:
    resp, pred, ok, structured = evaluate(prob, expected)
    if ok: score += 1
    tag = '✅' if ok else '❌'
    fmt = '🏷️' if structured else '🔢'
    print(f'{tag} {fmt} Problem: {prob}')
    print(f'   Expected: {expected} | Got: {pred}')
    print(f'   Response: {resp[:120]}')
    print()

print('='*55)
print(f'FINAL SCORE: {score}/{len(test_cases)} ({100*score//len(test_cases)}%)')
print('='*55)